In [8]:
import cv2
import numpy as np
import os

# ===== YOUR PATHS =====
DATASET_PATH = r"C:\PCB_Project\PCB_DATASET"
OUTPUT_PATH  = r"C:\PCB_Project\module1_output"

CATEGORIES = [
    "Missing_hole", "Mouse_bite", "Open_circuit",
    "Short", "Spur", "Spurious_copper"
]

TEMPLATE_DIR = os.path.join(DATASET_PATH, "PCB_USED")

In [9]:
def find_best_template(test_img):

    gray_test = cv2.cvtColor(test_img, cv2.COLOR_BGR2GRAY)
    best_score = -1
    best_temp_path = None

    test_small = cv2.resize(gray_test, (256, 256))

    for temp_name in os.listdir(TEMPLATE_DIR):
        temp_path = os.path.join(TEMPLATE_DIR, temp_name)

        temp_img = cv2.imread(temp_path, 0)
        if temp_img is None:
            continue

        temp_small = cv2.resize(temp_img, (256, 256))

        result = cv2.matchTemplate(
            test_small, temp_small,
            cv2.TM_CCOEFF_NORMED
        )

        _, max_val, _, _ = cv2.minMaxLoc(result)

        if max_val > best_score:
            best_score = max_val
            best_temp_path = temp_path

    return best_temp_path

In [15]:
def get_intense_mask(test_img, temp_img):

    gray_test = cv2.cvtColor(test_img, cv2.COLOR_BGR2GRAY)
    gray_temp = cv2.cvtColor(temp_img, cv2.COLOR_BGR2GRAY)

    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    gray_test = clahe.apply(gray_test)
    gray_temp = clahe.apply(gray_temp)

    diff = cv2.absdiff(gray_temp, gray_test)

    mask = cv2.adaptiveThreshold(
        diff, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        11, 2
    )

    mask = cv2.medianBlur(mask, 3)

    _, certainty_mask = cv2.threshold(diff, 30, 255, cv2.THRESH_BINARY)
    final_mask = cv2.bitwise_and(mask, certainty_mask)

    return final_mask

In [16]:
def process_module1():

    os.makedirs(os.path.join(OUTPUT_PATH, "defect_masks"), exist_ok=True)
    os.makedirs(os.path.join(OUTPUT_PATH, "sample_visuals"), exist_ok=True)

    print("🔥 Running Module 1...")

    for category in CATEGORIES:

        img_dir = os.path.join(DATASET_PATH, "images", category)

        for filename in os.listdir(img_dir):

            test_img = cv2.imread(os.path.join(img_dir, filename))
            if test_img is None:
                continue

            temp_path = find_best_template(test_img)
            temp_img = cv2.imread(temp_path)

            temp_img = cv2.resize(
                temp_img,
                (test_img.shape[1], test_img.shape[0])
            )

            mask = get_intense_mask(test_img, temp_img)

            out_name = f"{category}_{filename}"

            cv2.imwrite(
                os.path.join(OUTPUT_PATH, "defect_masks", out_name),
                mask
            )

            # Visualization
            h, w = 450, 450
            viz = np.hstack((
                cv2.resize(temp_img, (w, h)),
                cv2.resize(test_img, (w, h)),
                cv2.cvtColor(cv2.resize(mask, (w, h)), cv2.COLOR_GRAY2BGR)
            ))

            cv2.imwrite(
                os.path.join(OUTPUT_PATH, "sample_visuals", out_name),
                viz
            )

    print("✅ Module 1 DONE")

In [18]:
MASK_PATH = os.path.join(OUTPUT_PATH, "defect_masks")
OUTPUT_ROI_PATH = r"C:\PCB_Project\module2_output\cropped_defects"
VISUAL_PATH = r"C:\PCB_Project\module2_output\contour_visuals"

In [19]:
def process_module2():

    for cat in CATEGORIES:
        os.makedirs(os.path.join(OUTPUT_ROI_PATH, cat), exist_ok=True)

    os.makedirs(VISUAL_PATH, exist_ok=True)

    PADDING = 20

    print("🚀 Running Module 2...")

    for filename in os.listdir(MASK_PATH):

        mask = cv2.imread(os.path.join(MASK_PATH, filename), 0)

        # Extract category from filename
        sorted_categories = sorted(CATEGORIES, key=lambda x: -len(x))
        category = next(
            (cat for cat in sorted_categories if filename.startswith(cat + "_")),
            None
        )

        if category is None:
            continue

        orig_filename = filename.replace(f"{category}_", "")
        orig_img_path = os.path.join(
            DATASET_PATH, "images", category, orig_filename
        )

        orig_img = cv2.imread(orig_img_path)

        if orig_img is None or mask is None:
            continue

        contours, _ = cv2.findContours(
            mask,
            cv2.RETR_EXTERNAL,
            cv2.CHAIN_APPROX_SIMPLE
        )

        viz_img = orig_img.copy()

        for i, cnt in enumerate(contours):

            if cv2.contourArea(cnt) < 5:
                continue

            x, y, w, h = cv2.boundingRect(cnt)

            cv2.rectangle(viz_img, (x,y), (x+w,y+h), (0,0,255), 2)

            y1 = max(0, y-PADDING)
            y2 = min(orig_img.shape[0], y+h+PADDING)
            x1 = max(0, x-PADDING)
            x2 = min(orig_img.shape[1], x+w+PADDING)

            roi = orig_img[y1:y2, x1:x2]

            roi_name = f"{category}_{orig_filename.split('.')[0]}_roi_{i}.jpg"

            cv2.imwrite(
                os.path.join(OUTPUT_ROI_PATH, category, roi_name),
                roi
            )

        cv2.imwrite(
            os.path.join(VISUAL_PATH, f"viz_{filename}"),
            viz_img
        )

    print("✅ Module 2 DONE")

In [20]:
process_module1()
process_module2()

🔥 Running Module 1...
✅ Module 1 DONE
🚀 Running Module 2...
✅ Module 2 DONE
